# Fashion MNIST 服饰分类——机器学习算法对比研究

## 项目简介
本实验基于 Fashion MNIST 数据集（Zalando 服饰图片，10 类，70,000 张 28×28 灰度图），
分别采用 **传统机器学习方法（SVM、随机森林）** 与 **深度学习方法（MLP、CNN）** 进行分类，
并横向比较四种方法在准确率、训练时间、混淆矩阵等方面的表现。

**数据来源：** [Kaggle - Fashion MNIST](https://www.kaggle.com/datasets/zalando-research/fashionmnist)

**分工说明：**
- 组员A：数据探索、预处理、SVM + 随机森林
- 组员B：MLP + CNN 深度学习模型设计、训练与调参
- 共同完成：方法对比分析、可视化、报告撰写

---
## 1. 导入依赖库

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# 传统机器学习
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score,
                             confusion_matrix, classification_report)

# 深度学习
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

# 中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print(f'TensorFlow: {tf.__version__}')
gpu_count = len(tf.config.list_physical_devices('GPU'))
print(f'GPU Available: {gpu_count > 0} ({gpu_count} device(s))')

---
## 2. 数据加载与探索性分析（EDA）

In [ ]:
# 加载 Fashion MNIST 数据
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# 类别名称
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
class_names_cn = ['T恤', '裤子', '套头衫', '连衣裙', '外套',
                  '凉鞋', '衬衫', '运动鞋', '包', '短靴']

print(f'训练集: {x_train.shape}, 标签: {y_train.shape}')
print(f'测试集: {x_test.shape}, 标签: {y_test.shape}')
print(f'像素值范围: [{x_train.min()}, {x_train.max()}]')
print(f'类别数: {len(np.unique(y_train))}')

In [ ]:
# 随机展示训练集样本
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    idx = np.random.randint(0, len(x_train))
    ax.imshow(x_train[idx], cmap='gray')
    ax.set_title(class_names_cn[y_train[idx]], fontsize=9)
    ax.axis('off')
plt.suptitle('Fashion MNIST 训练集样本展示', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 训练集类别分布
unique, counts = np.unique(y_train, return_counts=True)
fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = plt.cm.tab10(np.arange(10))
bars = ax.bar(range(10), counts, color=colors_bar)
ax.set_xticks(range(10))
ax.set_xticklabels(class_names_cn, rotation=45, ha='right')
ax.set_ylabel('样本数量')
ax.set_title('训练集各类别样本数量分布', fontsize=14, fontweight='bold')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
            str(count), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 各类别平均图像（观察类内共性）
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    mean_img = x_train[y_train == i].mean(axis=0)
    ax.imshow(mean_img, cmap='gray')
    ax.set_title(f'{class_names_cn[i]}（均值）', fontsize=10)
    ax.axis('off')
plt.suptitle('各类别像素均值图像', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. 数据预处理

In [ ]:
# 扁平化（传统ML方法使用，将28×28展平为784维向量）
x_train_flat = x_train.reshape(x_train.shape[0], -1) / 255.0
x_test_flat = x_test.reshape(x_test.shape[0], -1) / 255.0

# 标准化（SVM等对特征尺度敏感的方法需要）
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train_flat)
x_test_scaled = scaler.transform(x_test_flat)

# 深度学习用：保持28×28，归一化并增加通道维度
x_train_cnn = x_train.astype('float32') / 255.0
x_test_cnn = x_test.astype('float32') / 255.0
x_train_cnn = x_train_cnn[..., np.newaxis]
x_test_cnn = x_test_cnn[..., np.newaxis]

# One-hot编码（深度学习分类用）
y_train_onehot = keras.utils.to_categorical(y_train, 10)
y_test_onehot = keras.utils.to_categorical(y_test, 10)

# 传统ML方法抽样（SVM在全部60k样本上训练太慢）
n_samples_ml = 10000
rng = np.random.RandomState(42)
idx = rng.choice(len(x_train_scaled), n_samples_ml, replace=False)
x_train_ml = x_train_scaled[idx]
y_train_ml = y_train[idx]

x_test_ml = x_test_scaled
y_test_ml = y_test

print(f'传统ML训练数据（抽样）: {x_train_ml.shape}')
print(f'深度学习训练数据: {x_train_cnn.shape}')
print('预处理完成。')

---
## 4. 方法一：SVM（支持向量机）

**原理：** 通过寻找最大间隔超平面进行分类，使用 RBF 核函数将数据映射到高维空间，使非线性可分数据在高维空间中线性可分。

**负责：组员A**

In [ ]:
print('训练 SVM（RBF核，C=10）...')
t0 = time.time()

svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm.fit(x_train_ml, y_train_ml)

t_train_svm = time.time() - t0

t0 = time.time()
y_pred_svm = svm.predict(x_test_ml)
t_pred_svm = time.time() - t0

acc_svm = accuracy_score(y_test_ml, y_pred_svm)
print(f'SVM 测试准确率: {acc_svm:.4f}')
print(f'训练时间: {t_train_svm:.2f}s | 预测时间: {t_pred_svm:.2f}s')
print()
print(classification_report(y_test_ml, y_pred_svm, target_names=class_names_cn))

In [ ]:
# SVM 混淆矩阵
cm_svm = confusion_matrix(y_test_ml, y_pred_svm)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names_cn, yticklabels=class_names_cn, ax=ax)
ax.set_xlabel('预测标签')
ax.set_ylabel('真实标签')
ax.set_title(f'SVM 混淆矩阵（准确率: {acc_svm:.2%}）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. 方法二：随机森林（Random Forest）

**原理：** 集成多棵决策树，通过 Bagging 自助采样和随机特征子集选择构建基学习器，投票决定最终分类结果。对高维数据鲁棒性好，可输出特征重要性。

**负责：组员A**

In [ ]:
print('训练随机森林（200棵树，最大深度20）...')
t0 = time.time()

rf = RandomForestClassifier(n_estimators=200, max_depth=20,
                            min_samples_split=5, n_jobs=-1, random_state=42)
rf.fit(x_train_ml, y_train_ml)

t_train_rf = time.time() - t0

t0 = time.time()
y_pred_rf = rf.predict(x_test_ml)
t_pred_rf = time.time() - t0

acc_rf = accuracy_score(y_test_ml, y_pred_rf)
print(f'随机森林 测试准确率: {acc_rf:.4f}')
print(f'训练时间: {t_train_rf:.2f}s | 预测时间: {t_pred_rf:.2f}s')
print()
print(classification_report(y_test_ml, y_pred_rf, target_names=class_names_cn))

In [ ]:
# 随机森林混淆矩阵
cm_rf = confusion_matrix(y_test_ml, y_pred_rf)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names_cn, yticklabels=class_names_cn, ax=ax)
ax.set_xlabel('预测标签')
ax.set_ylabel('真实标签')
ax.set_title(f'随机森林 混淆矩阵（准确率: {acc_rf:.2%}）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 随机森林特征重要性（还原为28×28热力图，观察哪些像素对分类贡献大）
importance = rf.feature_importances_.reshape(28, 28)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(importance, cmap='hot')
ax.set_title('随机森林 像素重要性热力图', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

---
## 6. 方法三：MLP（多层感知机）

**原理：** 全连接前馈神经网络，通过多层非线性变换（ReLU激活）逐层提取高阶特征，使用 Dropout 和 Batch Normalization 防止过拟合，Softmax 输出类别概率。

**负责：组员B**

In [ ]:
# 构建 MLP 模型
mlp_model = models.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

mlp_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
mlp_model.summary()

In [ ]:
# 训练 MLP
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

t0 = time.time()

history_mlp = mlp_model.fit(
    x_train_cnn, y_train_onehot,
    batch_size=128,
    epochs=30,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

t_train_mlp = time.time() - t0

# 评估
t0 = time.time()
mlp_loss, mlp_acc = mlp_model.evaluate(x_test_cnn, y_test_onehot, verbose=0)
t_pred_mlp = time.time() - t0

print(f'\nMLP 测试准确率: {mlp_acc:.4f}')
print(f'训练时间: {t_train_mlp:.2f}s | 预测时间: {t_pred_mlp:.4f}s')

In [ ]:
# MLP 训练曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_mlp.history['accuracy'], label='训练集', linewidth=2)
axes[0].plot(history_mlp.history['val_accuracy'], label='验证集', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('准确率')
axes[0].set_title('MLP 准确率曲线', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_mlp.history['loss'], label='训练集', linewidth=2)
axes[1].plot(history_mlp.history['val_loss'], label='验证集', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('损失')
axes[1].set_title('MLP 损失曲线', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# MLP 混淆矩阵
y_pred_mlp = mlp_model.predict(x_test_cnn, verbose=0).argmax(axis=1)
cm_mlp = confusion_matrix(y_test, y_pred_mlp)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_mlp, annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_names_cn, yticklabels=class_names_cn, ax=ax)
ax.set_xlabel('预测标签')
ax.set_ylabel('真实标签')
ax.set_title(f'MLP 混淆矩阵（准确率: {mlp_acc:.2%}）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. 方法四：CNN（卷积神经网络）

**原理：** 利用卷积核提取局部空间特征（边缘、纹理），池化层进行下采样，多层堆叠后通过全局平均池化输入全连接分类器。CNN 通过参数共享和局部连接大幅减少参数量，是图像分类的主流方法。

**负责：组员B**

In [ ]:
# 构建 CNN 模型（3个卷积块 + 全局池化 + 分类器）
cnn_model = models.Sequential([
    # Block 1
    layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(28, 28, 1)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Block 2
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Block 3
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # 分类器
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

cnn_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
cnn_model.summary()

In [ ]:
# 数据增强（轻微旋转、缩放、平移，提升泛化能力）
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
])

early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6)

t0 = time.time()

history_cnn = cnn_model.fit(
    data_augmentation(x_train_cnn), y_train_onehot,
    batch_size=64,
    epochs=50,
    validation_data=(x_test_cnn, y_test_onehot),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

t_train_cnn = time.time() - t0

# 评估
t0 = time.time()
cnn_loss, cnn_acc = cnn_model.evaluate(x_test_cnn, y_test_onehot, verbose=0)
t_pred_cnn = time.time() - t0

print(f'\nCNN 测试准确率: {cnn_acc:.4f}')
print(f'训练时间: {t_train_cnn:.2f}s | 预测时间: {t_pred_cnn:.4f}s')

In [ ]:
# CNN 训练曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_cnn.history['accuracy'], label='训练集', linewidth=2)
axes[0].plot(history_cnn.history['val_accuracy'], label='验证集', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('准确率')
axes[0].set_title('CNN 准确率曲线', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_cnn.history['loss'], label='训练集', linewidth=2)
axes[1].plot(history_cnn.history['val_loss'], label='验证集', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('损失')
axes[1].set_title('CNN 损失曲线', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# CNN 混淆矩阵
y_pred_cnn = cnn_model.predict(x_test_cnn, verbose=0).argmax(axis=1)
cm_cnn = confusion_matrix(y_test, y_pred_cnn)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Reds',
            xticklabels=class_names_cn, yticklabels=class_names_cn, ax=ax)
ax.set_xlabel('预测标签')
ax.set_ylabel('真实标签')
ax.set_title(f'CNN 混淆矩阵（准确率: {cnn_acc:.2%}）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# CNN 分类错误案例分析
errors = y_pred_cnn != y_test
error_indices = np.where(errors)[0]

print(f'CNN 错误分类样本数: {len(error_indices)} / {len(y_test)}')

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    if i < min(len(error_indices), 18):
        idx = error_indices[i]
        ax.imshow(x_test[idx], cmap='gray')
        ax.set_title(f'真实:{class_names_cn[y_test[idx]]}\n预测:{class_names_cn[y_pred_cnn[idx]]}', fontsize=8)
    ax.axis('off')
plt.suptitle('CNN 分类错误样本（前18个）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. 四种方法横向对比

In [ ]:
# 计算各类别的 F1 分数（f1_score 已在 cell-2 导入）
f1_svm = f1_score(y_test_ml, y_pred_svm, average='macro')
f1_rf  = f1_score(y_test_ml, y_pred_rf, average='macro')
f1_mlp = f1_score(y_test, y_pred_mlp, average='macro')
f1_cnn = f1_score(y_test, y_pred_cnn, average='macro')

# 汇总四种方法的结果
print('模型结果汇总：')
print(f'  SVM (CPU) : Acc={acc_svm:.4f}, F1={f1_svm:.4f}, Train={t_train_svm:.1f}s')
print(f'  RF  (CPU) : Acc={acc_rf:.4f}, F1={f1_rf:.4f}, Train={t_train_rf:.1f}s')
print(f'  MLP (GPU) : Acc={mlp_acc:.4f}, F1={f1_mlp:.4f}, Train={t_train_mlp:.1f}s')
print(f'  CNN (GPU) : Acc={cnn_acc:.4f}, F1={f1_cnn:.4f}, Train={t_train_cnn:.1f}s')

# 汇总表
df_results = pd.DataFrame({
    '方法':        ['SVM', '随机森林', 'MLP', 'CNN'],
    '准确率':      [acc_svm, acc_rf, mlp_acc, cnn_acc],
    'Macro F1':    [f1_svm, f1_rf, f1_mlp, f1_cnn],
    '训练时间(s)':  [t_train_svm, t_train_rf, t_train_mlp, t_train_cnn],
    '预测时间(s)':  [t_pred_svm, t_pred_rf, t_pred_mlp, t_pred_cnn],
})
df_results['准确率(%)'] = (df_results['准确率'] * 100).round(2)
df_results['Macro F1(%)'] = (df_results['Macro F1'] * 100).round(2)

df_results[['方法', '准确率(%)', 'Macro F1(%)', '训练时间(s)', '预测时间(s)']]

In [ ]:
# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors_bar = ["#3498db", "#2ecc71", "#f39c12", "#e74c3c"]
methods = df_results["方法"].values

# 准确率对比
bars = axes[0].bar(methods, df_results["准确率(%)"].values, color=colors_bar)
axes[0].set_ylabel("准确率 (%)")
axes[0].set_title("四种方法准确率对比", fontsize=13, fontweight="bold")
axes[0].set_ylim(70, 100)
for bar, val in zip(bars, df_results["准确率(%)"]):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                f"{val:.1f}%", ha="center", fontsize=11, fontweight="bold")

# Macro F1 对比
bars = axes[1].bar(methods, df_results["Macro F1(%)"].values, color=colors_bar)
axes[1].set_ylabel("Macro F1 (%)")
axes[1].set_title("四种方法 Macro F1 对比", fontsize=13, fontweight="bold")
axes[1].set_ylim(70, 100)
for bar, val in zip(bars, df_results["Macro F1(%)"]):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                f"{val:.1f}%", ha="center", fontsize=11, fontweight="bold")

# 训练时间对比
bars = axes[2].bar(methods, df_results["训练时间(s)"].values, color=colors_bar)
axes[2].set_ylabel("训练时间 (秒)")
axes[2].set_title("四种方法训练时间对比", fontsize=13, fontweight="bold")
for bar, val in zip(bars, df_results["训练时间(s)"]):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f"{val:.1f}s", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# 四个混淆矩阵并排展示
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

cms = [cm_svm, cm_rf, cm_mlp, cm_cnn]
titles = [
    f'SVM (Acc={acc_svm:.2%})',
    f'随机森林 (Acc={acc_rf:.2%})',
    f'MLP (Acc={mlp_acc:.2%})',
    f'CNN (Acc={cnn_acc:.2%})'
]
cmaps = ['Blues', 'Greens', 'Oranges', 'Reds']

for ax, cm, title, cmap in zip(axes.flat, cms, titles, cmaps):
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=class_names_cn, yticklabels=class_names_cn, ax=ax,
                cbar=False)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('预测标签')
    ax.set_ylabel('真实标签')

plt.suptitle('四种方法混淆矩阵对比', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 9. t-SNE 特征可视化

使用 t-SNE 将 CNN 倒数第二层提取的 256 维特征降维到 2D，可视化模型学到的特征空间分布，观察各类别的可分性。

In [ ]:
from sklearn.manifold import TSNE

# 提取 CNN 倒数第二层（Dense 256）的输出特征
feature_model = models.Model(inputs=cnn_model.input, outputs=cnn_model.layers[-3].output)
features = feature_model.predict(x_test_cnn[:2000], verbose=0)

# t-SNE 降维
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
features_2d = tsne.fit_transform(features)

# 可视化
fig, ax = plt.subplots(figsize=(12, 10))
scatter_colors = plt.cm.tab10(np.arange(10))
for i in range(10):
    mask = y_test[:2000] == i
    ax.scatter(features_2d[mask, 0], features_2d[mask, 1],
              c=[scatter_colors[i]], label=class_names_cn[i], s=10, alpha=0.7)
ax.legend(markerscale=3, fontsize=9)
ax.set_title("t-SNE 可视化：CNN 特征空间（测试集前2000样本）", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 10. 结论

### 主要发现

1. **CNN 准确率最高**：卷积神经网络在图像分类任务上表现最佳，准确率达到 **92.79%**，Macro F1 为 **92.77%**。卷积操作能有效提取图像的局部空间特征（边缘、纹理），加上数据增强的辅助，泛化能力远超全连接方法。在 RTX 4070 Ti 上训练耗时 128.2s。

2. **MLP 次之但结构简单**：多层感知机准确率 **88.98%**，Macro F1 **88.99%**，证明了深度非线性变换的有效性。但因为将 28x28 图像扁平化为 784 维向量后丢失了空间结构信息，效果比 CNN 低了约 3.8 个百分点。GPU 训练耗时 41.5s。

3. **随机森林表现稳健**：在不使用 GPU 加速的情况下，随机森林以仅 **1.91s** 的训练时间取得了 **85.26%** 的准确率，且能输出像素级特征重要性热力图，具有较好的可解释性。

4. **SVM 适用于小样本场景**：SVM（RBF 核，C=10）在 10,000 样本上准确率达到 **87.01%**（Macro F1 86.93%），略优于随机森林，但预测时间最长（25.9s），且扩展到全量 60,000 样本时训练时间将呈 O(n²)~O(n³) 增长，不适合大规模数据。

### 四种方法综合对比

| 方法 | 准确率 | Macro F1 | 训练时间 | 预测时间 | 运行设备 |
|------|--------|----------|----------|----------|----------|
| SVM | 87.01% | 86.93% | 10.95s | 25.89s | CPU (i7-13700) |
| 随机森林 | 85.26% | 85.04% | 1.91s | 0.12s | CPU (i7-13700) |
| MLP | 88.98% | 88.99% | 41.5s | 0.11s | GPU (RTX 4070 Ti) |
| CNN | 92.79% | 92.77% | 128.2s | 0.25s | GPU (RTX 4070 Ti) |

### 方法选型建议

| 实际场景 | 推荐方法 | 理由 |
|----------|----------|------|
| 追求最高精度 | CNN | 卷积核提取空间特征，准确率 92.79% |
| 计算资源有限 | 随机森林 | CPU 友好，训练仅需 1.91s 即达 85.26% |
| 需要可解释性 | 随机森林 | 像素级特征重要性直观可读 |
| 快速原型验证 | MLP | 结构简单，调参方便，准确率 88.98% |
| 小样本高精度 | SVM | 10k 样本上 87.01%，优于随机森林 |

### 误差分析

观察混淆矩阵可以发现，各类方法的共同难点集中在以下几对易混淆类别：
- **T恤 vs 衬衫**：两类外观高度相似（短袖 vs 长袖），SVM 对衬衫召回率仅 63%，随机森林仅 52%，MLP/CNN 对此有明显改善
- **套头衫 vs 外套**：均为上衣类，区分需要更细粒度的纹理特征
- **运动鞋 vs 短靴**：CNN 凭借卷积核的局部感受野能较好地区分鞋类，但传统方法对此存在一定混淆

### 未来改进方向

- 采用更深/更现代的 CNN 架构（ResNet、EfficientNet）进一步提升精度，目标突破 95%
- 引入集成学习（Voting / Stacking）融合 CNN + MLP + 随机森林的优势
- 使用 Optuna 或 Keras Tuner 进行自动超参数搜索
- 针对易混淆类别（Shirt vs T-shirt/Top）做 CutMix / MixUp 等针对性数据增强
- 探索 Vision Transformer（ViT）等新型架构在 Fashion MNIST 上的表现

---
*本实验完整代码已上传至 GitHub：https://github.com/wakawakajie/Fashion-MNIST-Classification*